In [5]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

In [6]:
class MyState(MessagesState):
    # messages: list 는 상속받아왔다!
    jpn: str
    eng: str
    chn: str

llm = init_chat_model('openai:gpt-4.1', temperature=1.3)

def jpn_node(state: MyState):
    prompt = '''
    너는 일본어 번역을 해주는 ai야.
    사용자의 입력을 일본 정서에 맞게 잘 번역해줘.
    '''
    result = llm.invoke([
        SystemMessage(prompt),
        state['messages'][-1]
    ])
    return {'jpn': result.content}

def eng_node(state: MyState):
    prompt = '''
    너는 영어 번역을 해주는 ai야.
    사용자의 입력을 미국 정서에 맞게 잘 번역해줘.
    '''
    result = llm.invoke([
        SystemMessage(prompt),
        state['messages'][-1]
    ])
    return {'eng': result.content}

def chn_node(state: MyState):
    prompt = '''
    너는 중국어 번역을 해주는 ai야.
    사용자의 입력을 중국 정서에 맞게 잘 번역해줘.
    '''
    result = llm.invoke([
        SystemMessage(prompt),
        state['messages'][-1]
    ])
    return {'chn': result.content}

def aggr_node(state: MyState):
    prompt = f'''
너는 다음 내용들을 잘 정리해서 보기 쉽게 만들어줘.

원문: {state['messages'][-1].content}
---
영어: {state['eng']}
---
일본어: {state['jpn']}
---
중국어: {state['chn']}
'''
    result = llm.invoke(prompt)  # AIMessage 저장
    return {'messages': [result]}

In [ ]:
builder = StateGraph(MyState)
builder.add_node(eng_node)
builder.add_node(jpn_node)
builder.add_node(chn_node)
builder.add_node(aggr_node)
builder.add_edge(START, 'eng_node')
builder.add_edge(START, 'jpn_node')
builder.add_edge(START, 'chn_node')
builder.add_edge('eng_node', 'aggr_node')
builder.add_edge('chn_node', 'aggr_node')
builder.add_edge('jpn_node', 'aggr_node')
builder.add_edge('aggr_node', END)

graph = builder.compile()
graph

In [ ]:
result = graph.invoke({
  'messages' : '안녕나는 조성훈이야. 만나서 반가워'
})


In [17]:
print(result)

{'messages': [HumanMessage(content='안녕나는 조성훈이야. 만나서 반가워', additional_kwargs={}, response_metadata={}, id='0e6731ae-a2ed-4a37-b7b1-6d42e986c9a9'), AIMessage(content="아래와 같이 보기 쉽게 정리해드릴게요.\n\n---\n\n**원문 (한국어)**  \n안녕 나는 조성훈이야. 만나서 반가워\n\n**영어**  \nHi, I'm Seonghoon Cho. Nice to meet you!\n\n**일본어**  \nこんにちは、チョ・ソンフンと申します。お会いできて嬉しいです。\n\n**중국어**  \n你好，我是赵成勋。很高兴认识你。\n\n---", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 106, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_5f99b03e83', 'id': 'chatcmpl-EJu6LIYasaYbhRDqGK8EY63h3SeJ2', 'service_tier

In [20]:
print(result['messages'])
print(result['messages'][-1])
print(result['messages'][-1].content)

[HumanMessage(content='안녕나는 조성훈이야. 만나서 반가워', additional_kwargs={}, response_metadata={}, id='0e6731ae-a2ed-4a37-b7b1-6d42e986c9a9'), AIMessage(content="아래와 같이 보기 쉽게 정리해드릴게요.\n\n---\n\n**원문 (한국어)**  \n안녕 나는 조성훈이야. 만나서 반가워\n\n**영어**  \nHi, I'm Seonghoon Cho. Nice to meet you!\n\n**일본어**  \nこんにちは、チョ・ソンフンと申します。お会いできて嬉しいです。\n\n**중국어**  \n你好，我是赵成勋。很高兴认识你。\n\n---", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 106, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_5f99b03e83', 'id': 'chatcmpl-EJu6LIYasaYbhRDqGK8EY63h3SeJ2', 'service_tier': 'default',